In [50]:
# hoping to test out implementation with multiple classes here
from transformers import pipeline
import pandas as pd
import os
from scipy.special import softmax

os.chdir("/Users/connorrust/Library/CloudStorage/Box-Box/Covid Policies/Data")
data = pd.read_csv("MN_Health_short.csv")

# extracting text from data
text = data.pop("Text").str.slice(0,100)
lst = text.to_list()

hypothesis_template = "This text is about {}"
classes_verbalized = ["economic relief", "reopening", "jobs", "housing", "vaccines","testing", "positive cases", 
                      "healthcare professionals", "healthcare infrastructure", "other", "research", "food"]

zeroshot_classifier = pipeline("zero-shot-classification", 
                               model="mlburnham/Political_DEBATE_DeBERTa_large_v1.1", 
                               device = "mps")  # change the model identifier here

output = zeroshot_classifier(lst, classes_verbalized, hypothesis_template=hypothesis_template, multi_label=True)

clean_output = []

for dct in output:
    nd = {}
    nd["sequence"] = dct["sequence"]
    for idx, label in enumerate(dct["labels"]):
        nd[label] = dct["scores"][idx]
    clean_output.append(nd)

df = pd.DataFrame(clean_output)

# mask to identify the columns with probabilities
prob_cols = df.columns.difference(['sequence'])
# creating a copy to apply softmax to
df2 = df.copy(deep=False) 
# Applying softmax to the copy
df2[prob_cols] = softmax(df[prob_cols].values, axis=1)

# combining probabilities with the original data
output =pd.concat([data, df2], axis=1)

df.to_csv('MAtopics_ZS_Burnham_modified_prompt_100_500chr.csv')



Device set to use mps


In [51]:
output

,Title,Date,State,Agency,sequence,vaccines,healthcare infrastructure,testing,healthcare professionals,other,reopening,positive cases,research,housing,food,jobs,economic relief
0,Minnesota announces COVID-19 vaccines fully d...,2021-01-04,MN,Governor,The Minnesota Department of Health today annou...,0.198143,0.072922,0.072897,0.072894,0.072893,0.072893,0.072893,0.072893,0.072893,0.072893,0.072893,0.072893
1,COVID-19 variant found in Minnesota - MN Dept...,2021-01-09,MN,Governor,The Minnesota Department of Health (MDH) today...,0.072929,0.072895,0.198112,0.072895,0.072896,0.072895,0.072900,0.072896,0.072895,0.072895,0.072895,0.072895
2,MDH lab testing confirms nation’s first known...,2021-01-25,MN,Governor,This case marks the first documented instance ...,0.158429,0.058283,0.058283,0.058283,0.058285,0.058283,0.158429,0.158428,0.058283,0.058448,0.058283,0.058283
3,Clusters of B117 variant COVID-19 cases in Ca...,2021-03-05,MN,Governor,A rapidly growing outbreak of variant COVID-19...,0.072960,0.072891,0.072914,0.072896,0.072889,0.072887,0.198127,0.072887,0.072887,0.072887,0.072887,0.072887
4,Travel-related COVID-19 cases prompt health o...,2021-04-16,MN,Governor,As health officials continue to report more ca...,0.064859,0.064861,0.175264,0.064862,0.064862,0.064860,0.176135,0.064859,0.064859,0.064859,0.064859,0.064859
5,Officials urge regular COVID-19 testing as ke...,2021-04-22,MN,Governor,Reflecting ongoing concerns about the potentia...,0.083329,0.083329,0.083329,0.083329,0.083377,0.083329,0.083329,0.083334,0.083329,0.083329,0.083329,0.083329
6,Minnesota announces new Midway COVID-19 Commu...,2022-05-11,MN,Governor,May 12 will be the last day of testing at the ...,0.132033,0.132020,0.132036,0.048574,0.048574,0.132036,0.048574,0.048574,0.048573,0.131861,0.048573,0.048573
7,State announces partnership with health plans...,2021-05-14,MN,Governor,Participating health plans starting this work ...,0.083245,0.084304,0.083245,0.083245,0.083247,0.083245,0.083245,0.083245,0.083245,0.083245,0.083245,0.083245
8,Lab testing confirms state’s first COVID-19 c...,2021-12-02,MN,Governor,The Minnesota Department of Health (MDH) today...,0.064847,0.175905,0.176120,0.064791,0.064792,0.064791,0.064792,0.064798,0.064791,0.064791,0.064791,0.064791


In [2]:
df

num_cols = df.columns.difference(['sequence'])
df2 = df.copy(deep=False)  
df2[num_cols] = softmax(df[num_cols].values, axis=1)
df2

,sequence,covid,other
0,The Minnesota Department of Health today annou...,0.999999,1.050766e-06
1,The Minnesota Department of Health (MDH) today...,1.000000,5.361989e-07
2,This case marks the first documented instance ...,0.445633,5.543672e-01
3,A rapidly growing outbreak of variant COVID-19...,0.999999,1.202849e-06
4,As health officials continue to report more ca...,0.999999,1.416599e-06
5,Reflecting ongoing concerns about the potentia...,0.026314,9.736865e-01
6,May 12 will be the last day of testing at the ...,0.999999,1.086696e-06
7,Participating health plans starting this work ...,0.309488,6.905116e-01
8,The Minnesota Department of Health (MDH) today...,0.999992,7.737041e-06
